# Task

**The LU decomposition and rank-one updates.**

```python
def lu(A):
    # return L, U with A = L @ U (no row swaps)
    ...

def solve_lu(L, U, b):
    # solve L U x = b by forward and back substitution
    ...
```
- Implement `lu` by the elimination of Theorem (see `thm:lu` in the [lecture notes](https://github.com/ArystanK/ml_school_math/blob/main/Lecture1/lecture1.pdf)): for each column, store the multipliers in $L$ and replace the rest of the matrix by its Schur complement. Check it on Example (see `ex:lu` in the [lecture notes](https://github.com/ArystanK/ml_school_math/blob/main/Lecture1/lecture1.pdf)), and check that `L @ U` equals $A$ for random $6 \times 6$ matrices.
- Implement `solve_lu`, and compare it with `np.linalg.solve` for random systems. For $A = \begin{pmatrix} 10^{-17} & 1 \\ 1 & 1 \end{pmatrix}$ and $\mathbf{b} = (1, 2)^\top$, the solution is very close to $(1, 1)$. What does your solver return, and why? Swapping the two rows first (partial pivoting) fixes the problem.
- For a random $200 \times 50$ data matrix $X$, compute $(X^\top X)^{-1}$. Then add $100$ new random rows one at a time, updating the inverse with the Sherman--Morrison formula of Theorem (see `thm:woodbury` in the [lecture notes](https://github.com/ArystanK/ml_school_math/blob/main/Lecture1/lecture1.pdf)), and compare the result with $(X^\top X)^{-1}$ computed from scratch for the final matrix. Compare the running times of the two approaches.


In [ ]:
import time
import numpy as np

np.set_printoptions(precision=5, suppress=True)
rng = np.random.default_rng(42)

In [ ]:
def rref(A, tol=1e-10):
    """Return the reduced row echelon form and zero-based pivot columns."""
    R = np.array(A, dtype=float, copy=True)
    if R.ndim != 2:
        raise ValueError("A must be a two-dimensional array")
    m, n = R.shape
    pivots = []
    row = 0
    for col in range(n):
        if row == m:
            break
        pivot = row + np.argmax(np.abs(R[row:, col]))
        if abs(R[pivot, col]) < tol:
            continue
        R[[row, pivot]] = R[[pivot, row]]
        R[row] /= R[row, col]
        for i in range(m):
            if i != row:
                R[i] -= R[i, col] * R[row]
        pivots.append(col)
        row += 1
    R[np.abs(R) < tol] = 0.0
    return R, pivots


def dual_basis(P):
    P = np.asarray(P, dtype=float)
    if P.ndim != 2 or P.shape[0] != P.shape[1]:
        raise ValueError("P must be square")
    n = P.shape[0]
    R, pivots = rref(np.hstack((P, np.eye(n))))
    if pivots[:n] != list(range(n)):
        raise ValueError("P must be invertible")
    return R[:, n:]


In [ ]:
def lu(A):
    U = np.array(A, dtype=float, copy=True)
    if U.ndim != 2 or U.shape[0] != U.shape[1]:
        raise ValueError("A must be square")
    n = U.shape[0]
    L = np.eye(n)
    for k in range(n):
        if U[k, k] == 0:
            raise ValueError("Zero pivot; row swapping is required")
        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i, k:] -= L[i, k] * U[k, k:]
    return L, U

In [ ]:
# Check LU decomposition on the example and random matrices.
A = np.array([[2., 1, 1], [4, 3, 3], [8, 7, 9]])
L, U = lu(A)
print("L =\n", L, "\nU =\n", U)
print("L @ U =\n", L @ U)
for i in range(10):
    A = rng.normal(size=(6, 6))
    L, U = lu(A)
    print(f"Matrix {i + 1}: maximum reconstruction error = {np.max(np.abs(L @ U - A)):.2e}")

In [ ]:
def solve_lu(L, U, b):
    b = np.asarray(b, dtype=float)
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x

In [ ]:
# Compare solutions with NumPy.
for i in range(10):
    A = rng.normal(size=(6, 6))
    b = rng.normal(size=6)
    L, U = lu(A)
    error = np.max(np.abs(solve_lu(L, U, b) - np.linalg.solve(A, b)))
    print(f"System {i + 1}: maximum difference = {error:.2e}")

In [ ]:
# Compare solutions with and without a row swap.
A = np.array([[1e-17, 1.], [1., 1.]])
b = np.array([1., 2.])
L, U = lu(A)
without_swap = solve_lu(L, U, b)
L, U = lu(A[[1, 0]])
with_swap = solve_lu(L, U, b[[1, 0]])
print("Without swapping:", without_swap)
print("After swapping:  ", with_swap)
print("NumPy solution:  ", np.linalg.solve(A, b))

# A tiny pivot causes rounding errors; swapping makes the pivot 1.

In [ ]:
# Compare inverse updates with recomputation.
# Sherman?Morrison: (G + xx.T)^-1 = G^-1 - (G^-1 x x.T G^-1)/(1 + x.T G^-1 x).
X = rng.normal(size=(200, 50))
new_rows = rng.normal(size=(100, 50))
initial_inverse = dual_basis(X.T @ X)

start = time.perf_counter()
updated = initial_inverse.copy()
for x in new_rows:
    left = updated @ x
    right = x @ updated
    updated -= np.outer(left, right) / (1 + x @ left)
update_time = time.perf_counter() - start

final_X = np.vstack((X, new_rows))
start = time.perf_counter()
recomputed = dual_basis(final_X.T @ final_X)
recompute_time = time.perf_counter() - start
reference = np.linalg.inv(final_X.T @ final_X)
start = time.perf_counter()
gram = X.T @ X
for x in new_rows:
    gram += np.outer(x, x)
    repeated = dual_basis(gram)
repeated_time = time.perf_counter() - start
print("Maximum error against NumPy:", np.max(np.abs(updated - reference)))
print("Maximum difference between methods:", np.max(np.abs(updated-recomputed)))
print(f"100 sequential updates: {update_time:.6f} s")
print(f"100 inverse recomputations: {repeated_time:.6f} s")
print(f"One inverse from final data: {recompute_time:.6f} s")
# Sequential timings exclude the initial inverse; the final-only timing is separate.